# 🏛️ Day 02a: Facade & Proxy Patterns

---

## 🎯 What You'll Master Today
- **Facade**: simplify a complex subsystem behind one clean interface
- **Proxy**: control access to an object (lazy loading, caching, access control)
- Real-world: home theater systems, database connection pools, image loaders

---

## 🎭 The Analogies

**Facade** = Hotel concierge  
You say "I want dinner reservations." The concierge handles calling the restaurant, booking, arranging transport. You don't deal with each system.

**Proxy** = Security guard at a building  
Same entrance, same building. But the guard checks your badge before letting you in. The proxy controls access to the real thing.

---

In [ ]:
# ============================================
# 1. FACADE — Home Theater System
# ============================================

# Complex subsystems
class TV:
    def on(self): print("📺 TV turning on")
    def set_input(self, src): print(f"📺 Input set to {src}")

class SoundSystem:
    def on(self): print("🔊 Sound system on")
    def set_volume(self, vol): print(f"🔊 Volume set to {vol}")
    def set_mode(self, mode): print(f"🔊 Mode: {mode}")

class StreamingPlayer:
    def on(self): print("▶️ Player on")
    def login(self, service): print(f"▶️ Logged into {service}")
    def play(self, movie): print(f"▶️ Playing: {movie}")

class Lights:
    def dim(self, level): print(f"💡 Lights dimmed to {level}%")


# FACADE — one method does everything
class HomeTheaterFacade:
    def __init__(self):
        self._tv = TV()
        self._sound = SoundSystem()
        self._player = StreamingPlayer()
        self._lights = Lights()
    
    def watch_movie(self, movie: str):
        """Single call replaces 7 subsystem calls."""
        print("\n🎬 Setting up movie night...")
        self._lights.dim(20)
        self._tv.on()
        self._tv.set_input("HDMI1")
        self._sound.on()
        self._sound.set_volume(50)
        self._sound.set_mode("surround")
        self._player.on()
        self._player.login("Netflix")
        self._player.play(movie)
        print("🍿 Enjoy!\n")


# Client code — simple!
theater = HomeTheaterFacade()
theater.watch_movie("Interstellar")

In [ ]:
# ============================================
# 2. PROXY — Three Types
# ============================================
from abc import ABC, abstractmethod

# ── Virtual Proxy (Lazy Loading) ──
class Image(ABC):
    @abstractmethod
    def display(self) -> str: pass

class RealImage(Image):
    def __init__(self, filename: str):
        self._filename = filename
        self._load()  # expensive!
    
    def _load(self):
        print(f"  ⏳ Loading {self._filename} from disk (expensive!)")
    
    def display(self) -> str:
        return f"🖼️ Displaying {self._filename}"


class ImageProxy(Image):
    """Delays loading until display() is actually called."""
    def __init__(self, filename: str):
        self._filename = filename
        self._real_image = None  # not loaded yet!
    
    def display(self) -> str:
        if self._real_image is None:
            self._real_image = RealImage(self._filename)
        return self._real_image.display()


print("Creating proxy (no loading yet):")
img = ImageProxy("photo_8mb.jpg")
print("Proxy created. Image NOT loaded.\n")

print("First display (triggers load):")
print(img.display())

print("\nSecond display (cached):")
print(img.display())

In [ ]:
# ── Protection Proxy (Access Control) ──

class Database(ABC):
    @abstractmethod
    def query(self, sql: str) -> str: pass
    @abstractmethod
    def execute(self, sql: str) -> str: pass

class RealDatabase(Database):
    def query(self, sql: str) -> str:
        return f"📊 Results for: {sql}"
    def execute(self, sql: str) -> str:
        return f"✅ Executed: {sql}"

class DatabaseProxy(Database):
    """Checks permissions before allowing operations."""
    def __init__(self, db: RealDatabase, user_role: str):
        self._db = db
        self._role = user_role
    
    def query(self, sql: str) -> str:
        # Everyone can read
        return self._db.query(sql)
    
    def execute(self, sql: str) -> str:
        if self._role != "admin":
            return f"🚫 Access denied. Role '{self._role}' cannot execute writes."
        return self._db.execute(sql)


db = RealDatabase()
admin_db = DatabaseProxy(db, "admin")
viewer_db = DatabaseProxy(db, "viewer")

print(admin_db.query("SELECT * FROM users"))
print(admin_db.execute("DELETE FROM logs"))
print()
print(viewer_db.query("SELECT * FROM users"))
print(viewer_db.execute("DELETE FROM logs"))  # blocked!

In [ ]:
# ── Caching Proxy ──

class WeatherService:
    def get_forecast(self, city: str) -> str:
        print(f"  🌐 Calling external API for {city}...")
        return f"☀️ 28°C in {city}"

class CachingProxy:
    def __init__(self, service: WeatherService):
        self._service = service
        self._cache: dict[str, str] = {}
    
    def get_forecast(self, city: str) -> str:
        if city not in self._cache:
            self._cache[city] = self._service.get_forecast(city)
        else:
            print(f"  ⚡ Cache hit for {city}")
        return self._cache[city]


weather = CachingProxy(WeatherService())
print(weather.get_forecast("Mumbai"))     # API call
print(weather.get_forecast("Mumbai"))     # cached!
print(weather.get_forecast("Delhi"))      # API call

```
╔══════════════════════════════════════════════════════╗
║               PROXY TYPES CHEAT SHEET               ║
╠══════════════════╦═══════════════════════════════════╣
║  Virtual Proxy    ║  Lazy init — load on first use   ║
║  Protection Proxy ║  Access control — check perms    ║
║  Caching Proxy    ║  Cache results — avoid re-work   ║
║  Remote Proxy     ║  Local stand-in for remote obj   ║
║  Logging Proxy    ║  Log all calls before forwarding ║
╚══════════════════╩═══════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Facade vs Adapter? | Facade simplifies a complex subsystem. Adapter changes one interface to another |
| 2 | What problem does Proxy solve? | Controls access: lazy loading, permissions, caching, logging |
| 3 | Proxy vs Decorator? | Proxy controls access (identity matters). Decorator adds behavior (stacking matters) |
| 4 | When to use Facade? | When client shouldn't know about subsystem complexity. Simplify API surface |
| 5 | Real-world Proxy examples? | Django QuerySet (lazy), ORM lazy loading, Python's `__getattr__`, API rate limiters |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Facade = simplify (many subsystems → one call)        │
## │  • Proxy = control access (same interface, extra logic)  │
## │  • Virtual Proxy → lazy loading                          │
## │  • Protection Proxy → access control                     │
## │  • Caching Proxy → avoid redundant work                  │
## │  • Proxy has same interface as the real object            │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Composite & Flyweight** — tree structures and memory optimization